# 05 — Pro Consumidor Prices | Gold KPIs

Consumer-price KPIs per **month × product × presentation**, across retail segments, into `gold.consumer_price_metrics`.

| KPI | Column | Status |
|---|---|---|
| CON-001 Average price | `avg_price_dop` | Calculated |
| CON-002 Median price | `median_price_dop` | Calculated |
| CON-003 / 004 Min / max price | `min_price_dop`, `max_price_dop` (+ cheapest / most expensive segment) | Calculated |
| CON-005 Month-over-month % | `mom_change_pct`, `mom_segments_matched` | Like-for-like: average of the per-segment change for segments priced in both consecutive months (the segment mix varies, so the cross-segment average is not compared). NULL when no segment matches |
| CON-006 YoY %, CON-007 YTD %, CON-008 Volatility | — | **Not calculable**: the source covers only 3 months |
| CON-009 Regional difference % | — | **Not calculable**: the source has no geography; `price_spread_pct` (max vs min across segments) is provided instead and is not a regional measure |
| CON-010 Category index | — | Pending methodology |

Prices are only compared within the same product **and** presentation (e.g. "Galón 128 Oz"). **Status:** Development.

In [0]:
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.silver.proconsumidor_prices_monthly"
TARGET = f"{CATALOG}.gold.consumer_price_metrics"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

s = spark.table(SOURCE).withColumn("p", F.col("price_dop").cast("double"))
k = ["reference_month", "product", "presentation"]
agg = (s.groupBy(*k).agg(
        F.avg("p").alias("avg"), F.percentile_approx("p", 0.5, 10000).alias("median"),
        F.min("p").alias("mn"), F.max("p").alias("mx"), F.countDistinct("retail_segment").alias("segments"),
        F.min_by("retail_segment", "p").alias("cheapest_segment"), F.max_by("retail_segment", "p").alias("most_expensive_segment")))
# CON-005 like-for-like: the segment mix changes month to month, so the change is measured per segment
# present in both consecutive months and then averaged (matched sample), never on the cross-segment average.
ws = Window.partitionBy("product", "presentation", "retail_segment").orderBy("reference_month")
seg = (s.withColumn("prev_p", F.lag("p").over(ws)).withColumn("prev_m", F.lag("reference_month").over(ws))
        .filter(F.months_between("reference_month", "prev_m") == 1)
        .groupBy(*k).agg(F.avg((F.col("p") / F.col("prev_p") - 1) * 100).alias("mom"), F.count("*").alias("matched")))
g = agg.join(seg, k, "left").withColumn("matched", F.coalesce("matched", F.lit(0)))
d2 = lambda c: F.round(c, 2).cast("decimal(12,2)")
out = g.select(*k,
    d2(F.col("avg")).alias("avg_price_dop"), d2(F.col("median")).alias("median_price_dop"),
    d2(F.col("mn")).alias("min_price_dop"), d2(F.col("mx")).alias("max_price_dop"),
    "cheapest_segment", "most_expensive_segment", F.col("segments").cast("int").alias("segments_observed"),
    F.round((F.col("mx") / F.col("mn") - 1) * 100, 4).cast("decimal(12,4)").alias("price_spread_pct"),
    F.round(F.col("mom"), 4).cast("decimal(12,4)").alias("mom_change_pct"), F.col("matched").cast("int").alias("mom_segments_matched"),
    F.lit(RUN_ID).alias("_gold_run_id"), F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"))

checks = []
def add(name, rtype, tested, failed):
    checks.append((RUN_ID, TARGET, name, rtype, int(tested), int(failed), "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS))
n = out.count()
add("unique_month_product_presentation", "enforced", n, n - out.select(*k).distinct().count())
add("min_le_median_le_max", "enforced", n, out.filter((F.col("min_price_dop") > F.col("median_price_dop")) | (F.col("median_price_dop") > F.col("max_price_dop"))).count())
add("silver_rows_covered", "enforced", s.count(), abs(s.count() - out.agg(F.sum("segments_observed")).first()[0]))
for c in checks: print(f"  {c[6]:<4} {c[2]:<36} failed={c[5]:,}/{c[4]:,}")
assert not [c for c in checks if c[6] == "FAIL"]

out.orderBy("product", "presentation", "reference_month").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Gold: Pro Consumidor KPIs CON-001 to CON-005 per month, product and presentation across retail segments. CON-006 to CON-010 not calculable with current source coverage.'")
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)

top = spark.sql(f"""SELECT product, presentation, avg_price_dop, mom_change_pct, price_spread_pct, cheapest_segment, most_expensive_segment
  FROM {TARGET} WHERE reference_month = (SELECT MAX(reference_month) FROM {TARGET}) AND mom_change_pct IS NOT NULL ORDER BY mom_change_pct DESC LIMIT 5""")
display(top)
m = spark.sql(f"SELECT MIN(reference_month) a, MAX(reference_month) z, COUNT(DISTINCT product) p FROM {TARGET}").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "rows": n, "months": [str(m["a"]), str(m["z"])], "products": m["p"],
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks}, "top_mom": [{k2: str(v) for k2, v in r.asDict().items()} for r in top.collect()]}, ensure_ascii=False))